# UTKFace Cropped Images - Google Colab Kullanım Rehberi

Bu notebook, UTKFace veri setindeki kırpılmış (cropped) yüz görsellerini Google Colab üzerinde nasıl kullanacağınızı gösterir.

## UTKFace Veri Seti Hakkında

UTKFace, 20.000'den fazla yüz görselini içeren büyük ölçekli bir veri setidir. Her görsel yaş, cinsiyet ve etnik köken bilgilerini içerir.

### Dosya İsimlendirme Formatı
```
[yaş]_[cinsiyet]_[ırk]_[tarih&zaman].jpg
```

- **Yaş**: 0-116 arası tam sayı
- **Cinsiyet**: 0 (Erkek), 1 (Kadın)
- **Irk**: 0 (Beyaz), 1 (Siyah), 2 (Asyalı), 3 (Hint), 4 (Diğer)

Örnek: `39_1_0_20170116174525125.jpg` → 39 yaşında, kadın, beyaz

## Adım 1: Gerekli Kütüphaneleri Yükleme

In [ ]:
# Gerekli kütüphaneleri import edin
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO
import zipfile
import glob
from tqdm import tqdm

# Colab'da kullanılacak ek kütüphaneler
try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Google Colab")

print("Kütüphaneler başarıyla yüklendi!")

## Adım 2: Veri Setini İndirme

UTKFace veri setini indirmek için farklı yöntemler:

### Yöntem 1: Kaggle'dan İndirme (Önerilen)

In [ ]:
# Kaggle API kurulumu
!pip install -q kaggle

# Kaggle API token'ınızı yükleyin (kaggle.json dosyası)
# Kaggle hesabınızdan API token alın: https://www.kaggle.com/settings
if IN_COLAB:
    from google.colab import files
    print("Kaggle.json dosyanızı yükleyin:")
    uploaded = files.upload()
    
    # Kaggle klasörü oluştur ve token'ı kopyala
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# UTKFace veri setini indir
!kaggle datasets download -d jangedoo/utkface-new

# ZIP dosyasını çıkar
!unzip -q utkface-new.zip -d utkface_data/

print("Veri seti başarıyla indirildi ve çıkarıldı!")

### Yöntem 2: Google Drive'dan Yükleme

In [ ]:
# Google Drive'ı bağla
if IN_COLAB:
    drive.mount('/content/drive')
    
    # Eğer veri setiniz Google Drive'da mevcutsa
    # drive_path = '/content/drive/MyDrive/UTKFace/'
    # Bu yolu veri setinizin bulunduğu yere göre ayarlayın
    
    print("Google Drive başarıyla bağlandı!")

### Yöntem 3: Manuel Yükleme

In [ ]:
# Yerel bilgisayarınızdan dosya yükleyin
if IN_COLAB:
    print("ZIP dosyanızı yükleyin:")
    uploaded = files.upload()
    
    # ZIP dosyasını çıkar
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            !unzip -q {filename} -d utkface_data/
            print(f"{filename} çıkarıldı!")

## Adım 3: Veri Setini Keşfetme

In [ ]:
# Veri seti klasör yolunu belirleyin
data_dir = 'utkface_data/UTKFace/'  # Gerekirse bu yolu ayarlayın

# Tüm görsel dosyalarını listele
image_paths = glob.glob(os.path.join(data_dir, '*.jpg'))
print(f"Toplam görsel sayısı: {len(image_paths)}")

# İlk birkaç dosya ismini göster
print("\nÖrnek dosya isimleri:")
for path in image_paths[:5]:
    print(os.path.basename(path))

## Adım 4: Dosya İsimlerinden Metadata Çıkarma

In [ ]:
def parse_filename(filename):
    """
    UTKFace dosya isminden yaş, cinsiyet ve ırk bilgilerini çıkarır.
    
    Format: [yaş]_[cinsiyet]_[ırk]_[tarih].jpg
    """
    try:
        parts = filename.split('_')
        age = int(parts[0])
        gender = int(parts[1])
        race = int(parts[2])
        
        # Cinsiyet ve ırk etiketlerini anlamlı hale getir
        gender_label = 'Erkek' if gender == 0 else 'Kadın'
        race_labels = ['Beyaz', 'Siyah', 'Asyalı', 'Hint', 'Diğer']
        race_label = race_labels[race] if race < len(race_labels) else 'Bilinmeyen'
        
        return {
            'age': age,
            'gender': gender,
            'gender_label': gender_label,
            'race': race,
            'race_label': race_label
        }
    except:
        return None

# Test et
test_filename = os.path.basename(image_paths[0])
metadata = parse_filename(test_filename)
print(f"Dosya: {test_filename}")
print(f"Metadata: {metadata}")

## Adım 5: DataFrame Oluşturma

In [ ]:
# Tüm görseller için metadata içeren bir DataFrame oluştur
data_list = []

for img_path in tqdm(image_paths, desc="Dosyalar işleniyor"):
    filename = os.path.basename(img_path)
    metadata = parse_filename(filename)
    
    if metadata:
        data_list.append({
            'filename': filename,
            'filepath': img_path,
            'age': metadata['age'],
            'gender': metadata['gender_label'],
            'race': metadata['race_label']
        })

# DataFrame oluştur
df = pd.DataFrame(data_list)

print(f"\nDataFrame oluşturuldu: {len(df)} kayıt")
print("\nİlk birkaç kayıt:")
print(df.head())

print("\nVeri seti istatistikleri:")
print(df.describe())

## Adım 6: Veri Setini Görselleştirme

In [ ]:
# Yaş dağılımı
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
df['age'].hist(bins=50, edgecolor='black')
plt.title('Yaş Dağılımı')
plt.xlabel('Yaş')
plt.ylabel('Frekans')

# Cinsiyet dağılımı
plt.subplot(1, 3, 2)
df['gender'].value_counts().plot(kind='bar')
plt.title('Cinsiyet Dağılımı')
plt.xlabel('Cinsiyet')
plt.ylabel('Sayı')
plt.xticks(rotation=0)

# Irk dağılımı
plt.subplot(1, 3, 3)
df['race'].value_counts().plot(kind='bar')
plt.title('Irk Dağılımı')
plt.xlabel('Irk')
plt.ylabel('Sayı')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

## Adım 7: Rastgele Görselleri Görüntüleme

In [ ]:
def display_images(df, n=9, figsize=(15, 15)):
    """
    Rastgele n adet görseli görüntüler.
    """
    # Rastgele görsel seç
    random_samples = df.sample(n=n)
    
    # Grid oluştur
    rows = int(np.ceil(np.sqrt(n)))
    cols = int(np.ceil(n / rows))
    
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten() if n > 1 else [axes]
    
    for idx, (_, row) in enumerate(random_samples.iterrows()):
        if idx >= len(axes):
            break
            
        # Görseli yükle
        img = Image.open(row['filepath'])
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(f"Yaş: {row['age']}, {row['gender']}, {row['race']}",
                           fontsize=10)
    
    # Kullanılmayan eksenleri gizle
    for idx in range(n, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

# Rastgele 9 görsel göster
display_images(df, n=9)

## Adım 8: Filtrelenmiş Görselleri Görüntüleme

In [ ]:
# Örnek: 20-30 yaş arası kadınları filtrele
filtered_df = df[(df['age'] >= 20) & (df['age'] <= 30) & (df['gender'] == 'Kadın')]
print(f"Filtrelenmiş kayıt sayısı: {len(filtered_df)}")

# Filtrelenmiş görselleri göster
if len(filtered_df) > 0:
    display_images(filtered_df, n=min(9, len(filtered_df)))

## Adım 9: Görselleri Yükleme ve Ön İşleme

In [ ]:
def load_and_preprocess_image(filepath, target_size=(224, 224)):
    """
    Görseli yükler ve ön işleme yapar.
    
    Args:
        filepath: Görsel dosya yolu
        target_size: Hedef boyut (genişlik, yükseklik)
    
    Returns:
        numpy array: Normalleştirilmiş görsel
    """
    # Görseli yükle
    img = Image.open(filepath)
    
    # Boyutlandır
    img = img.resize(target_size)
    
    # Numpy array'e çevir
    img_array = np.array(img)
    
    # Normalize et (0-1 arası)
    img_array = img_array.astype('float32') / 255.0
    
    return img_array

# Test et
test_img = load_and_preprocess_image(df.iloc[0]['filepath'])
print(f"Görsel şekli: {test_img.shape}")
print(f"Piksel değer aralığı: [{test_img.min():.2f}, {test_img.max():.2f}]")

# Görseli görüntüle
plt.figure(figsize=(5, 5))
plt.imshow(test_img)
plt.axis('off')
plt.title('Ön İşlenmiş Görsel')
plt.show()

## Adım 10: Batch Halinde Veri Yükleme

In [ ]:
def create_data_generator(df, batch_size=32, target_size=(224, 224)):
    """
    Batch halinde veri üreteci oluşturur.
    
    Args:
        df: Veri DataFrame'i
        batch_size: Batch boyutu
        target_size: Görsel boyutu
    
    Yields:
        images: Görsel batch'i
        labels: Etiket batch'i (yaş, cinsiyet, ırk)
    """
    num_samples = len(df)
    indices = np.arange(num_samples)
    
    while True:
        # Karıştır
        np.random.shuffle(indices)
        
        for start_idx in range(0, num_samples, batch_size):
            end_idx = min(start_idx + batch_size, num_samples)
            batch_indices = indices[start_idx:end_idx]
            
            # Batch verileri
            batch_images = []
            batch_ages = []
            batch_genders = []
            batch_races = []
            
            for idx in batch_indices:
                row = df.iloc[idx]
                
                # Görseli yükle
                img = load_and_preprocess_image(row['filepath'], target_size)
                batch_images.append(img)
                
                # Etiketleri ekle
                batch_ages.append(row['age'])
                batch_genders.append(1 if row['gender'] == 'Kadın' else 0)
                batch_races.append(['Beyaz', 'Siyah', 'Asyalı', 'Hint', 'Diğer'].index(row['race']))
            
            yield (
                np.array(batch_images),
                {
                    'age': np.array(batch_ages),
                    'gender': np.array(batch_genders),
                    'race': np.array(batch_races)
                }
            )

# Generator'ı test et
gen = create_data_generator(df, batch_size=8)
images, labels = next(gen)

print(f"Batch görsel şekli: {images.shape}")
print(f"Yaş etiketleri: {labels['age']}")
print(f"Cinsiyet etiketleri: {labels['gender']}")
print(f"Irk etiketleri: {labels['race']}")

## Adım 11: Train/Validation/Test Ayrımı

In [ ]:
from sklearn.model_selection import train_test_split

# Veriyi ayır: %70 train, %15 validation, %15 test
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train set: {len(train_df)} görsel")
print(f"Validation set: {len(val_df)} görsel")
print(f"Test set: {len(test_df)} görsel")

# DataFrame'leri kaydet (opsiyonel)
train_df.to_csv('train_data.csv', index=False)
val_df.to_csv('val_data.csv', index=False)
test_df.to_csv('test_data.csv', index=False)

print("\nVeri setleri CSV olarak kaydedildi!")

## Adım 12: TensorFlow/Keras ile Kullanım Örneği

In [ ]:
# TensorFlow import et (Colab'da genellikle yüklüdür)
try:
    import tensorflow as tf
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    print(f"TensorFlow version: {tf.__version__}")
    
    # Data augmentation için ImageDataGenerator
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        horizontal_flip=True,
        zoom_range=0.2
    )
    
    val_datagen = ImageDataGenerator(rescale=1./255)
    
    print("\nTensorFlow data augmentation hazır!")
    
except ImportError:
    print("TensorFlow yüklü değil. Yüklemek için: !pip install tensorflow")

## Adım 13: PyTorch ile Kullanım Örneği

In [ ]:
# PyTorch kullanmak isterseniz
try:
    import torch
    from torch.utils.data import Dataset, DataLoader
    import torchvision.transforms as transforms
    
    print(f"PyTorch version: {torch.__version__}")
    
    class UTKFaceDataset(Dataset):
        def __init__(self, dataframe, transform=None):
            self.df = dataframe.reset_index(drop=True)
            self.transform = transform
        
        def __len__(self):
            return len(self.df)
        
        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            
            # Görseli yükle
            image = Image.open(row['filepath']).convert('RGB')
            
            # Transform uygula
            if self.transform:
                image = self.transform(image)
            
            # Etiketler
            age = torch.tensor(row['age'], dtype=torch.float32)
            gender = torch.tensor(1 if row['gender'] == 'Kadın' else 0, dtype=torch.long)
            race_map = {'Beyaz': 0, 'Siyah': 1, 'Asyalı': 2, 'Hint': 3, 'Diğer': 4}
            race = torch.tensor(race_map.get(row['race'], 4), dtype=torch.long)
            
            return image, {'age': age, 'gender': gender, 'race': race}
    
    # Transform tanımla
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Dataset ve DataLoader oluştur
    train_dataset = UTKFaceDataset(train_df, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    
    print(f"\nPyTorch Dataset oluşturuldu: {len(train_dataset)} örnek")
    print("DataLoader hazır!")
    
except ImportError:
    print("PyTorch yüklü değil. Yüklemek için: !pip install torch torchvision")

## Özet ve Sonraki Adımlar

Bu notebook ile UTKFace cropped images veri setini Google Colab'da kullanmayı öğrendiniz:

✅ Veri setini indirme ve yükleme
✅ Dosya isimlerinden metadata çıkarma
✅ Veri setini keşfetme ve görselleştirme
✅ Görselleri görüntüleme ve filtreleme
✅ Görsel ön işleme
✅ Batch halinde veri yükleme
✅ Train/Val/Test ayrımı
✅ TensorFlow/Keras ile kullanım
✅ PyTorch ile kullanım

### Sonraki Adımlar:

1. **Model Eğitimi**: Bu veri seti ile yaş tahmini, cinsiyet sınıflandırması veya ırk tanıma modelleri eğitebilirsiniz
2. **Transfer Learning**: ResNet, VGG, MobileNet gibi önceden eğitilmiş modelleri kullanabilirsiniz
3. **Data Augmentation**: Veri artırma teknikleri ile model performansını iyileştirebilirsiniz
4. **Multi-task Learning**: Aynı anda yaş, cinsiyet ve ırk tahmin eden modeller geliştirebilirsiniz

### Faydalı Kaynaklar:

- [UTKFace Dataset](https://susanqq.github.io/UTKFace/)
- [Kaggle UTKFace Dataset](https://www.kaggle.com/datasets/jangedoo/utkface-new)
- [TensorFlow Tutorials](https://www.tensorflow.org/tutorials)
- [PyTorch Tutorials](https://pytorch.org/tutorials/)